# IA102 — Notebook 3 — Réseaux convolutifs et données images

**Durée : 35 minutes** · Objectifs visés : **O2** (créer des réseaux), **O3** (entraîner avec PyTorch), **O4** (utiliser un GPU)

On entraîne un **réseau convolutif** (CNN) à reconnaître 10 catégories de vêtements à partir de 60 000 images en niveaux de gris.

### Plan
1. Le jeu de données FashionMNIST
2. Les limites des réseaux entièrement connectés pour les images
3. La convolution
4. Le sous-échantillonnage (pooling)
5. `Dataset` et `DataLoader` : l'entraînement par mini-lots
6. Architecture du CNN
7. Entraînement sur GPU ou CPU
8. Analyse des erreurs
9. Exercice : un réseau plus profond

In [ ]:
# Outil de questions de vérification (exécutez cette cellule sans la modifier)
import ipywidgets as widgets
from IPython.display import display, Markdown

def quiz(question, options, bonne, explication):
    """Affiche une question à choix multiples avec rétroaction immédiate."""
    choix = widgets.RadioButtons(options=options, value=None,
                                 layout={"width": "max-content"})
    bouton = widgets.Button(description="Vérifier", button_style="primary")
    sortie = widgets.Output()

    def verifier(_):
        with sortie:
            sortie.clear_output()
            if choix.value is None:
                display(Markdown("Veuillez choisir une réponse."))
            elif choix.value == options[bonne]:
                display(Markdown(f"**Bonne réponse.** {explication}"))
            else:
                display(Markdown("**Réponse incorrecte.** Relisez la section précédente et essayez de nouveau."))

    bouton.on_click(verifier)
    display(Markdown(f"**Question de vérification.** {question}"), choix, bouton, sortie)

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)

# Les nœuds de calcul de l'Alliance n'ont pas accès à Internet : téléchargez les données au préalable
# avec preparer_donnees.py (sur un nœud de connexion) et indiquez leur dossier dans IA102_DONNEES.
DOSSIER_DONNEES = os.environ.get("IA102_DONNEES", "donnees")
appareil = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil de calcul :", appareil)

## 1. Le jeu de données FashionMNIST

70 000 images de 28 × 28 pixels (60 000 pour l'entraînement, 10 000 pour le test), réparties en 10 classes. `transforms.ToTensor()` convertit chaque image en tenseur de forme (1, 28, 28), à valeurs dans [0, 1].

In [ ]:
transformation = transforms.ToTensor()
donnees_ent = datasets.FashionMNIST(DOSSIER_DONNEES, train=True, download=True, transform=transformation)
donnees_test = datasets.FashionMNIST(DOSSIER_DONNEES, train=False, download=True, transform=transformation)
CLASSES = ["T-shirt/haut", "Pantalon", "Pull", "Robe", "Manteau", "Sandale", "Chemise", "Basket", "Sac", "Bottine"]
print(f"Entraînement : {len(donnees_ent)} images | Test : {len(donnees_test)} images")

image, etiquette = donnees_ent[0]
print("Forme d'une image :", tuple(image.shape), "| étiquette :", etiquette, "->", CLASSES[etiquette])

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.ravel(), range(16)):
    img, lab = donnees_ent[i]
    ax.imshow(img[0], cmap="gray_r"); ax.set_title(CLASSES[lab], fontsize=8); ax.axis("off")
plt.suptitle("Exemples d'images FashionMNIST"); plt.show()

## 2. Les limites des réseaux entièrement connectés pour les images

On pourrait aplatir chaque image en un vecteur de 784 valeurs et utiliser le réseau du notebook 2. Deux difficultés apparaissent :

1. **Nombre de paramètres.** Pour une photographie couleur de 224 × 224 pixels, une seule couche de 512 neurones requiert 224 × 224 × 3 × 512 ≈ **77 millions** de poids.
2. **Perte de la structure spatiale.** Une fois l'image aplatie, la notion de pixels voisins disparaît. Un motif décalé de deux pixels active des poids complètement différents.

Le réseau convolutif répond à ces deux difficultés grâce à la **connectivité locale** et au **partage des poids**.

## 3. La convolution

Un **filtre** (ou noyau) est une petite matrice de poids, par exemple 3 × 3, que l'on fait glisser sur l'image. À chaque position, on calcule la somme des produits entre le filtre et les pixels recouverts :

$$S(i, j) = \sum_{u=-1}^{1} \sum_{v=-1}^{1} K(u, v)\, I(i + u,\, j + v) + b$$

Le résultat est une **carte de caractéristiques**, qui prend de fortes valeurs là où le motif recherché est présent. Le même filtre est appliqué partout : 9 poids suffisent pour détecter un motif quelle que soit sa position.

Appliquons des filtres définis à la main :

In [ ]:
bottine = donnees_ent[0][0].unsqueeze(0)          # forme (1, 1, 28, 28)
filtres = {
    "Contours verticaux":   [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "Contours horizontaux": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "Lissage":              [[1 / 9] * 3] * 3,
}
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
axes[0].imshow(bottine[0, 0], cmap="gray_r"); axes[0].set_title("Image originale")
for ax, (nom, noyau) in zip(axes[1:], filtres.items()):
    noyau = torch.tensor(noyau, dtype=torch.float32).view(1, 1, 3, 3)
    ax.imshow(F.conv2d(bottine, noyau, padding=1)[0, 0].abs(), cmap="magma"); ax.set_title(nom)
for ax in axes: ax.axis("off")
plt.suptitle("Chaque filtre met en évidence un motif différent"); plt.show()

Dans un CNN, les filtres **ne sont pas définis à la main** : leurs poids sont des paramètres appris par descente de gradient, comme ceux d'une couche linéaire.

### Exercice 6 — Concevoir un filtre

Définissez un filtre 3 × 3 (par exemple un filtre de rehaussement des détails, ou un détecteur de diagonales) et appliquez-le à une autre image.

In [ ]:
noyau_perso = [[..., ..., ...],
               [..., ..., ...],
               [..., ..., ...]]   # À COMPLÉTER

img = donnees_ent[1][0].unsqueeze(0)
carte = F.conv2d(img, torch.tensor(noyau_perso, dtype=torch.float32).view(1, 1, 3, 3), padding=1)
fig, (a, b) = plt.subplots(1, 2, figsize=(6, 3))
a.imshow(img[0, 0], cmap="gray_r"); a.set_title(CLASSES[donnees_ent[1][1]]); a.axis("off")
b.imshow(carte[0, 0].abs(), cmap="magma"); b.set_title("Carte de caractéristiques"); b.axis("off"); plt.show()

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
noyau_perso = [[ 0, -1,  0],
               [-1,  5, -1],
               [ 0, -1,  0]]   # rehaussement des détails

img = donnees_ent[1][0].unsqueeze(0)
carte = F.conv2d(img, torch.tensor(noyau_perso, dtype=torch.float32).view(1, 1, 3, 3), padding=1)
fig, (a, b) = plt.subplots(1, 2, figsize=(6, 3))
a.imshow(img[0, 0], cmap="gray_r"); a.set_title(CLASSES[donnees_ent[1][1]]); a.axis("off")
b.imshow(carte[0, 0].abs(), cmap="magma"); b.set_title("Carte de caractéristiques"); b.axis("off"); plt.show()
```

</details>

## 4. Le sous-échantillonnage (pooling)

Le **max pooling** 2 × 2 divise la carte en blocs de 4 valeurs et conserve le maximum de chacun. La taille est divisée par deux dans chaque direction ; l'information « le motif est présent dans cette zone » est conservée, avec une tolérance aux petits décalages.

In [ ]:
carte = torch.tensor([[1., 3., 2., 0.], [4., 2., 1., 5.], [0., 1., 3., 2.], [2., 6., 1., 1.]])
resume = F.max_pool2d(carte.view(1, 1, 4, 4), kernel_size=2)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
couleurs = np.array([[0, 0, 1, 1], [0, 0, 1, 1], [2, 2, 3, 3], [2, 2, 3, 3]])
for ax, m, coul, titre in [(axes[0], carte, couleurs, "Carte 4 × 4"), (axes[1], resume[0, 0], np.array([[0, 1], [2, 3]]), "Après max pooling 2 × 2")]:
    ax.imshow(coul, cmap="Pastel1", vmin=0, vmax=8)
    for (i, j), v in np.ndenumerate(m.numpy()):
        ax.text(j, i, int(v), ha="center", va="center", fontsize=14, fontweight="bold")
    ax.set_title(titre); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 5. `Dataset` et `DataLoader` : l'entraînement par mini-lots

On ne présente ni l'ensemble des 60 000 images à la fois (mémoire insuffisante), ni une image à la fois (calcul inefficace). On utilise des **mini-lots**, par exemple de 64 images : une mise à jour des paramètres par lot. C'est la **descente de gradient stochastique par mini-lots**.

- Un **`Dataset`** donne accès à une observation par son indice (`dataset[i]` renvoie l'image et son étiquette).
- Un **`DataLoader`** regroupe les observations en lots, les **mélange** à chaque époque (`shuffle=True`) et peut les charger en parallèle (`num_workers`, voir notebook 5).

In [ ]:
charge_ent = DataLoader(donnees_ent, batch_size=64, shuffle=True)
charge_test = DataLoader(donnees_test, batch_size=1000)
images, etiquettes = next(iter(charge_ent))
print("Lot d'images :", tuple(images.shape), "-> (lot, canaux, hauteur, largeur)")
print("Lot d'étiquettes :", tuple(etiquettes.shape))
print("Nombre de lots par époque :", len(charge_ent))

Pour des données personnelles (fichiers d'images, tableaux, signaux), on écrit une sous-classe de `Dataset` avec deux méthodes :

```python
class MonJeuDeDonnees(torch.utils.data.Dataset):
    def __init__(self, chemins, etiquettes, transformation=None):
        self.chemins, self.etiquettes, self.transformation = chemins, etiquettes, transformation
    def __len__(self):
        return len(self.chemins)
    def __getitem__(self, i):
        image = Image.open(self.chemins[i])
        if self.transformation:
            image = self.transformation(image)
        return image, self.etiquettes[i]
```

## 6. Architecture du CNN

Deux blocs **convolution → ReLU → max pooling** extraient des caractéristiques ; une petite partie entièrement connectée effectue la classification.

```
Image (1, 28, 28)
 -> Conv 3×3, 16 filtres, ReLU, MaxPool 2×2   -> (16, 14, 14)   motifs simples : contours, coins
 -> Conv 3×3, 32 filtres, ReLU, MaxPool 2×2   -> (32, 7, 7)     motifs composés : col, semelle, manche
 -> Flatten                                    -> 1568 valeurs
 -> Linear 1568 -> 64, ReLU, Linear 64 -> 10   -> un score par classe
```

Plus on progresse dans le réseau, plus les caractéristiques sont abstraites et couvrent une grande zone de l'image (champ récepteur).

In [ ]:
class CNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.extracteur = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifieur = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifieur(self.extracteur(x))

modele = CNN().to(appareil)
print(modele)
print("Nombre de paramètres :", sum(p.numel() for p in modele.parameters()))

**Prédiction.** Combien de paramètres compterait un réseau entièrement connecté 784 → 512 → 10 ? Plus ou moins que ce CNN ?

<details><summary><b>Afficher la réponse</b></summary>

784 × 512 + 512 + 512 × 10 + 10 = **407 050** paramètres, soit environ quatre fois plus que ce CNN, pour une exactitude généralement inférieure.
</details>

## 7. Entraînement sur GPU ou CPU

Pour calculer sur GPU, **le modèle et les données doivent se trouver sur le même appareil** : on déplace le modèle une fois (`modele.to(appareil)`), puis chaque lot au moment de son utilisation.

### Exercice 7 — Boucle par mini-lots

In [ ]:
def entrainer_une_epoque(modele, chargeur, optimiseur, perte_fn):
    modele.train()
    total = 0.0
    for images, etiquettes in chargeur:
        images, etiquettes = ..., ...      # À COMPLÉTER : transférer le lot vers l'appareil
        sortie = ...                       # À COMPLÉTER : les cinq étapes
        perte = ...
        ...
        ...
        ...
        total += perte.item() * len(images)
    return total / len(chargeur.dataset)

def evaluer(modele, chargeur, perte_fn):
    modele.eval()
    total, correctes = 0.0, 0
    with torch.no_grad():
        for images, etiquettes in chargeur:
            images, etiquettes = images.to(appareil), etiquettes.to(appareil)
            sortie = modele(images)
            total += perte_fn(sortie, etiquettes).item() * len(images)
            correctes += (sortie.argmax(dim=1) == etiquettes).sum().item()
    return total / len(chargeur.dataset), correctes / len(chargeur.dataset)

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
def entrainer_une_epoque(modele, chargeur, optimiseur, perte_fn):
    modele.train()
    total = 0.0
    for images, etiquettes in chargeur:
        images, etiquettes = images.to(appareil), etiquettes.to(appareil)
        sortie = modele(images)
        perte = perte_fn(sortie, etiquettes)
        optimiseur.zero_grad()
        perte.backward()
        optimiseur.step()
        total += perte.item() * len(images)
    return total / len(chargeur.dataset)

def evaluer(modele, chargeur, perte_fn):
    modele.eval()
    total, correctes = 0.0, 0
    with torch.no_grad():
        for images, etiquettes in chargeur:
            images, etiquettes = images.to(appareil), etiquettes.to(appareil)
            sortie = modele(images)
            total += perte_fn(sortie, etiquettes).item() * len(images)
            correctes += (sortie.argmax(dim=1) == etiquettes).sum().item()
    return total / len(chargeur.dataset), correctes / len(chargeur.dataset)
```

</details>

Entraînement sur **3 époques** : environ une minute sur un processeur, quelques secondes sur un GPU.

In [ ]:
torch.manual_seed(0)
modele = CNN().to(appareil)
optimiseur = torch.optim.Adam(modele.parameters(), lr=1e-3)
perte_fn = nn.CrossEntropyLoss()
suivi = {"perte_ent": [], "perte_test": [], "exactitude_test": []}

for epoque in range(1, 4):
    debut = time.time()
    perte_ent = entrainer_une_epoque(modele, charge_ent, optimiseur, perte_fn)
    perte_test, exact_test = evaluer(modele, charge_test, perte_fn)
    suivi["perte_ent"].append(perte_ent); suivi["perte_test"].append(perte_test); suivi["exactitude_test"].append(exact_test)
    print(f"Époque {epoque} | perte entraînement {perte_ent:.3f} | perte test {perte_test:.3f} | "
          f"exactitude test {exact_test:.1%} | {time.time() - debut:.0f} s")

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3))
ep = range(1, len(suivi["perte_ent"]) + 1)
a.plot(ep, suivi["perte_ent"], "o-", label="Entraînement"); a.plot(ep, suivi["perte_test"], "o-", label="Test")
a.set_xlabel("Époque"); a.set_ylabel("Perte"); a.set_title("Perte"); a.set_xticks(list(ep)); a.legend()
b.plot(ep, suivi["exactitude_test"], "o-", color="#1e8449"); b.set_xlabel("Époque"); b.set_xticks(list(ep))
b.set_title("Exactitude sur le test"); plt.tight_layout(); plt.show()

## 8. Analyse des erreurs

L'exactitude globale ne révèle pas quelles classes sont confondues. La matrice de confusion (IA101, notebook 3) le montre.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

modele.eval(); vraies, predites, toutes_images = [], [], []
with torch.no_grad():
    for images, etiquettes in charge_test:
        vraies.append(etiquettes); predites.append(modele(images.to(appareil)).argmax(1).cpu()); toutes_images.append(images)
vraies, predites, toutes_images = torch.cat(vraies), torch.cat(predites), torch.cat(toutes_images)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(confusion_matrix(vraies, predites), display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=60)
ax.set_xlabel("Prédiction"); ax.set_ylabel("Réalité"); ax.set_title("Matrice de confusion du CNN"); plt.show()

In [ ]:
erreurs = torch.where(vraies != predites)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for ax, i in zip(axes.ravel(), erreurs):
    ax.imshow(toutes_images[i, 0], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"Réel : {CLASSES[vraies[i]]}\nPrédit : {CLASSES[predites[i]]}", fontsize=8, color="#c0392b")
plt.suptitle("Exemples d'erreurs"); plt.tight_layout(); plt.show()

**Discussion.** Les confusions les plus fréquentes (chemise, T-shirt, pull, manteau) sont-elles plausibles ? Un observateur humain les commettrait-il sur des images de 28 × 28 pixels ?

## 9. Exercice : un réseau plus profond

### Exercice 8 — Ajouter un troisième bloc convolutif

Ajoutez un bloc `Conv2d(32, 64) → ReLU → MaxPool2d(2)`. Après trois sous-échantillonnages, la carte passe de 7 × 7 à **3 × 3** (division entière) : l'entrée de la première couche linéaire devient `64 * 3 * 3`.

**Prédiction.** L'exactitude va-t-elle augmenter ? Et le temps par époque ?

In [ ]:
class CNNProfond(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.extracteur = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            ...   # À COMPLÉTER
        )
        self.classifieur = nn.Sequential(nn.Flatten(), nn.Linear(..., 64), nn.ReLU(), nn.Linear(64, n_classes))   # À COMPLÉTER

    def forward(self, x):
        return self.classifieur(self.extracteur(x))

torch.manual_seed(0)
profond = CNNProfond().to(appareil)
opt_profond = torch.optim.Adam(profond.parameters(), lr=1e-3)
for epoque in range(1, 4):
    debut = time.time()
    entrainer_une_epoque(profond, charge_ent, opt_profond, perte_fn)
    print(f"Époque {epoque} | exactitude test {evaluer(profond, charge_test, perte_fn)[1]:.1%} | {time.time() - debut:.0f} s")
print(f"Référence, CNN à 2 blocs : {suivi['exactitude_test'][-1]:.1%}")

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
class CNNProfond(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.extracteur = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifieur = nn.Sequential(nn.Flatten(), nn.Linear(64 * 3 * 3, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.classifieur(self.extracteur(x))

torch.manual_seed(0)
profond = CNNProfond().to(appareil)
opt_profond = torch.optim.Adam(profond.parameters(), lr=1e-3)
for epoque in range(1, 4):
    debut = time.time()
    entrainer_une_epoque(profond, charge_ent, opt_profond, perte_fn)
    print(f"Époque {epoque} | exactitude test {evaluer(profond, charge_test, perte_fn)[1]:.1%} | {time.time() - debut:.0f} s")
print(f"Référence, CNN à 2 blocs : {suivi['exactitude_test'][-1]:.1%}")
```

</details>

Ajouter des couches n'améliore pas automatiquement les résultats : avec trois époques, le réseau plus profond obtient un résultat comparable. Une capacité accrue demande généralement plus de données, plus d'époques ou une régularisation (notebook 4).

### Sauvegarder le modèle

In [ ]:
torch.save(modele.state_dict(), "cnn_fashion.pt")       # on sauvegarde les paramètres, pas le code
recharge = CNN(); recharge.load_state_dict(torch.load("cnn_fashion.pt", map_location="cpu"))
print("Paramètres sauvegardés puis rechargés.")

In [ ]:
quiz("Pourquoi un CNN compte-t-il beaucoup moins de paramètres qu'un réseau entièrement connecté sur une image ?",
     ["Parce qu'il réduit la résolution des images", "Parce qu'un même petit filtre est appliqué à toutes les positions (partage des poids)", "Parce qu'il n'a pas de biais"],
     1, "Un filtre 3 × 3 possède 9 poids, quel que soit le nombre de positions où il est appliqué.")

In [ ]:
quiz("Erreur « Expected all tensors to be on the same device ». Que faut-il faire ?",
     ["Réinstaller PyTorch", "Placer le modèle et chaque lot de données sur le même appareil avec .to(appareil)", "Réduire la taille des lots"],
     1, "Le modèle est déplacé une fois ; chaque lot est déplacé au moment de son utilisation.")

## À retenir

- La **convolution** applique un filtre appris à toutes les positions de l'image (connectivité locale, partage des poids).
- Le **max pooling** réduit la taille des cartes en conservant l'information essentielle.
- Un CNN empile des blocs **Conv → ReLU → Pool**, puis `Flatten` et des couches `Linear`.
- `Dataset` donne accès aux observations ; `DataLoader` produit des **mini-lots** mélangés.
- Calcul sur **GPU** : `modele.to(appareil)` une fois, `lot.to(appareil)` à chaque lot.

**Notebook suivant :** 4 — Améliorer la généralisation.